# PathSnap HMM and Viterbi Decoder — Implementation Notebook

This notebook contains a reusable sequence decoder. A hidden state is a candidate road arc for one GPS observation. Emission scores come from the classifier; transition scores combine road-network connectivity with the distance between consecutive GPS observations.

The synthetic example is only a code demonstration. Replace its candidate probabilities and geometry adapter with the outputs from your team’s model before reporting real results. Do not present synthetic metrics as experiment results.


## 1. Imports and decoder settings

In [ ]:
from collections import defaultdict
import math
import numpy as np

EPS = 1e-12


## 2. Transition model

Road arcs are directed pairs `(from_node, to_node)`. We precompute shortest directed network distances up to a configurable number of hops. A transition is feasible when the route distance between candidate arcs is compatible with observed GPS displacement plus a tolerance. Infeasible transitions receive a strong penalty, rather than silently deleting all paths.

This is a starter modeling choice: document and tune its parameters using training and student-test data only.

In [ ]:
def build_arc_graph(arcs):
    """Map arc_id -> (from_node, to_node), preserving row-index arc IDs."""
    return {i: (int(row[0]), int(row[1])) for i, row in enumerate(arcs)}

def shortest_node_distances(graph, start, max_hops=6):
    """BFS distances in hop units from a node, limited for runtime."""
    dist = {start: 0}
    queue = [start]
    for node in queue:
        if dist[node] >= max_hops:
            continue
        for nxt in graph.get(node, ()):
            if nxt not in dist:
                dist[nxt] = dist[node] + 1
                queue.append(nxt)
    return dist

def transition_score(prev_arc, curr_arc, arc_endpoints, node_graph,
                     observed_distance_m, max_hops=6, meters_per_hop=80.0,
                     distance_tolerance_m=150.0, disconnected_log_penalty=-12.0,
                     distance_scale_m=150.0):
    """Log transition score; node connectivity plus rough distance consistency."""
    if prev_arc not in arc_endpoints or curr_arc not in arc_endpoints:
        return disconnected_log_penalty
    _, prev_end = arc_endpoints[prev_arc]
    curr_start, _ = arc_endpoints[curr_arc]
    if prev_arc == curr_arc:
        network_hops = 0
    else:
        d = shortest_node_distances(node_graph, prev_end, max_hops=max_hops)
        if curr_start not in d:
            return disconnected_log_penalty
        network_hops = d[curr_start]
    # Convert network hops to a rough distance. Calibrate to your network scale.
    network_distance = network_hops * meters_per_hop
    excess = max(0.0, network_distance - observed_distance_m - distance_tolerance_m)
    shortfall = max(0.0, observed_distance_m - network_distance - distance_tolerance_m)
    mismatch = excess + shortfall
    return -mismatch / max(distance_scale_m, EPS)

def viterbi_decode(candidate_arc_ids, emission_probabilities, arc_endpoints,
                   node_graph, observed_distances_m=None, *,
                   distance_scale_m=150.0, meters_per_hop=80.0,
                   distance_tolerance_m=150.0, max_hops=6,
                   disconnected_log_penalty=-12.0):
    """Decode one sequence. Inputs are lists per observation, aligned by candidate.

    Empty candidate rows are represented as [-1] with probability [1.0]; that
    unknown state lets the sequence continue, while transitions through it are
    penalized. Returns one chosen candidate ID per observation.
    """
    if len(candidate_arc_ids) != len(emission_probabilities):
        raise ValueError("Candidate and probability sequences have different lengths")
    n = len(candidate_arc_ids)
    if n == 0:
        return []
    if observed_distances_m is None:
        observed_distances_m = [0.0] * n
    if len(observed_distances_m) != n:
        raise ValueError("observed_distances_m must have one value per observation")

    states, emits = [], []
    for ids, probs in zip(candidate_arc_ids, emission_probabilities):
        if len(ids) != len(probs):
            raise ValueError("Each candidate row must align with its probability row")
        if not ids:
            ids, probs = [-1], [1.0]
        probs = np.asarray(probs, dtype=float)
        probs = np.nan_to_num(probs, nan=0.0, posinf=0.0, neginf=0.0)
        probs = np.clip(probs, 0.0, None)
        if probs.sum() <= 0:
            probs = np.ones(len(probs), dtype=float) / len(probs)
        else:
            probs /= probs.sum()
        states.append(list(map(int, ids)))
        emits.append(np.log(np.maximum(probs, EPS)))

    dp = [emits[0].copy()]
    back = [np.full(len(states[0]), -1, dtype=int)]
    for t in range(1, n):
        cur_scores = np.full(len(states[t]), -np.inf)
        cur_back = np.full(len(states[t]), -1, dtype=int)
        for j, curr in enumerate(states[t]):
            for i, prev in enumerate(states[t-1]):
                if prev == -1 or curr == -1:
                    trans = disconnected_log_penalty
                else:
                    trans = transition_score(
                        prev, curr, arc_endpoints, node_graph, observed_distances_m[t],
                        max_hops=max_hops, meters_per_hop=meters_per_hop,
                        distance_tolerance_m=distance_tolerance_m,
                        disconnected_log_penalty=disconnected_log_penalty,
                        distance_scale_m=distance_scale_m)
                score = dp[t-1][i] + trans + emits[t][j]
                if score > cur_scores[j]:
                    cur_scores[j], cur_back[j] = score, i
        dp.append(cur_scores)
        back.append(cur_back)

    path_idx = [int(np.argmax(dp[-1]))]
    for t in range(n-1, 0, -1):
        path_idx.append(int(back[t][path_idx[-1]]))
    path_idx.reverse()
    return [states[t][j] for t, j in enumerate(path_idx)]


## 3. GPS displacement and route continuity helpers

Haversine distance approximates distance between consecutive GPS observations. Route continuity here is the fraction of consecutive predicted arcs that connect directly (end node equals next start node). This strict measure can undercount valid multi-arc transitions when observations are sparse; report that limitation and optionally also report reachability within a hop limit.

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    r = 6_371_000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*r*math.asin(math.sqrt(a))

def gps_step_distances(track):
    coords = track[["lon", "lat"]].to_numpy(dtype=float)
    out = [0.0]
    for i in range(1, len(coords)):
        out.append(haversine_m(*coords[i-1], *coords[i]))
    return out

def direct_route_continuity(predicted_arc_ids, arc_endpoints):
    pairs = [(a, b) for a, b in zip(predicted_arc_ids[:-1], predicted_arc_ids[1:]) if a >= 0 and b >= 0]
    if not pairs:
        return float("nan")
    return sum(arc_endpoints.get(a, (None, None))[1] == arc_endpoints.get(b, (None, None))[0]
               for a, b in pairs) / len(pairs)

def reachable_route_continuity(predicted_arc_ids, arc_endpoints, node_graph, max_hops=6):
    pairs = [(a, b) for a, b in zip(predicted_arc_ids[:-1], predicted_arc_ids[1:]) if a >= 0 and b >= 0]
    if not pairs:
        return float("nan")
    good = 0
    for a, b in pairs:
        end = arc_endpoints.get(a, (None, None))[1]
        start = arc_endpoints.get(b, (None, None))[0]
        if end is not None and start in shortest_node_distances(node_graph, end, max_hops):
            good += 1
    return good / len(pairs)


## 4. Synthetic smoke example (not a dataset result)

In [ ]:
# Directed chain: 0->1->2->3. Each observation has a plausible and a distracting candidate.
arcs_demo = [(0,1), (1,2), (2,3), (0,3)]
arc_endpoints_demo = build_arc_graph(arcs_demo)
node_graph_demo = defaultdict(list)
for u, v in arcs_demo:
    node_graph_demo[u].append(v)
candidates_demo = [[0,3], [1,3], [2,3]]
probs_demo = [[.55,.45], [.52,.48], [.55,.45]]
path_demo = viterbi_decode(candidates_demo, probs_demo, arc_endpoints_demo, node_graph_demo,
                           observed_distances_m=[0,75,75], meters_per_hop=75,
                           distance_tolerance_m=30)
print("Synthetic decoded arc IDs:", path_demo)
print("Direct continuity:", direct_route_continuity(path_demo, arc_endpoints_demo))


## 5. Integrate the team model

Expected per-trip inputs: candidate arc IDs and classifier probabilities for every observation, GPS track, `arcs` table, and reference arc ID per observation. The classifier teammate should provide candidate IDs and probabilities in matching row order. Adapt the following cell to their output schema; do not invent or manually adjust labels.

In [ ]:
# TODO: replace this adapter with your teammate's real model output loader.
# candidates_by_obs = [[arc_id_1, arc_id_2, ...], ...]
# probabilities_by_obs = [[p_1, p_2, ...], ...]  # same ordering as candidate IDs
# y_true = [...]  # reference arc ID aligned per GPS observation, for permitted dev data
# track = DataFrame with lon, lat; arcs = DataFrame with from_node, to_node

# After adapting the inputs, run:
# arc_endpoints = build_arc_graph(arcs[["from_node", "to_node"]].itertuples(index=False, name=None))
# node_graph = defaultdict(list)
# for u, v in arc_endpoints.values(): node_graph[u].append(v)
# y_classifier = [ids[int(np.argmax(probs))] if ids else -1
#                 for ids, probs in zip(candidates_by_obs, probabilities_by_obs)]
# decoded = viterbi_decode(candidates_by_obs, probabilities_by_obs, arc_endpoints, node_graph,
#                          gps_step_distances(track))


## 6. Compare classifier-only and HMM predictions

The metric cell expects aligned arc IDs per observation. Compute metrics per trip and aggregate across the 10 student-test trips; avoid pooling in a way that lets long trips dominate without also reporting per-trip results. Candidate miss rate is useful because a decoder cannot recover a true arc absent from the candidate set.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, log_loss
import pandas as pd

def compare_predictions(y_true, y_classifier, y_hmm, candidates_by_obs=None,
                       arc_endpoints=None, node_graph=None):
    y_true = np.asarray(y_true, dtype=int)
    y_classifier = np.asarray(y_classifier, dtype=int)
    y_hmm = np.asarray(y_hmm, dtype=int)
    if not (len(y_true) == len(y_classifier) == len(y_hmm)):
        raise ValueError("Truth and both prediction sequences must be aligned")
    out = []
    for name, pred in (("Classifier only", y_classifier), ("HMM/Viterbi", y_hmm)):
        out.append({"method": name, "top1_accuracy": accuracy_score(y_true, pred),
                    "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
                    "direct_continuity": direct_route_continuity(pred, arc_endpoints) if arc_endpoints else np.nan,
                    "reachable_continuity": reachable_route_continuity(pred, arc_endpoints, node_graph) if arc_endpoints and node_graph else np.nan})
    if candidates_by_obs is not None:
        miss = [truth not in row for truth, row in zip(y_true, candidates_by_obs)]
        print(f"Candidate miss rate: {np.mean(miss):.4f}")
    return pd.DataFrame(out)

# Run after integrating real development-trip arrays:
# compare_predictions(y_true, y_classifier, decoded, candidates_by_obs, arc_endpoints, node_graph)


## 7. Folium prediction map

Create one map per representative student-test trip. The predicted route is rendered by following predicted arc IDs and joining only shared endpoints; discontinuities start a new segment so the map does not imply a false connection.

In [ ]:
def route_segments(predicted_arc_ids, arc_endpoints, nodes):
    """Split arc geometry at discontinuities. Return lists of [lat, lon] points."""
    segments, current = [], []
    prev_end = None
    for arc_id in predicted_arc_ids:
        if arc_id < 0 or arc_id not in arc_endpoints:
            if len(current) > 1: segments.append(current)
            current, prev_end = [], None
            continue
        u, v = arc_endpoints[arc_id]
        if not (0 <= u < len(nodes) and 0 <= v < len(nodes)):
            continue
        start = [float(nodes.iloc[u].lat), float(nodes.iloc[u].lon)]
        end = [float(nodes.iloc[v].lat), float(nodes.iloc[v].lon)]
        if prev_end is not None and u != prev_end:
            if len(current) > 1: segments.append(current)
            current = []
        if not current: current = [start]
        current.append(end)
        prev_end = v
    if len(current) > 1: segments.append(current)
    return segments

def make_prediction_map(track, nodes, arcs, y_true, y_classifier, y_hmm, arc_endpoints, output_html):
    import folium
    m = folium.Map(location=[float(track.lat.mean()), float(track.lon.mean())], zoom_start=14, control_scale=True)
    gps = folium.FeatureGroup(name="GPS observations")
    for _, row in track.iterrows():
        folium.CircleMarker([row.lat,row.lon], radius=3, color="blue", fill=True, fill_opacity=.7).add_to(gps)
    gps.add_to(m)
    for label, preds, color, show in (("Reference route", y_true, "green", True),
                                       ("Classifier only", y_classifier, "orange", False),
                                       ("HMM/Viterbi", y_hmm, "red", True)):
        fg = folium.FeatureGroup(name=label, show=show)
        for seg in route_segments(preds, arc_endpoints, nodes):
            folium.PolyLine(seg, color=color, weight=4, opacity=.85, tooltip=label).add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(str(output_html))
    return m

# Once real arrays/data are loaded:
# prediction_map = make_prediction_map(track, nodes, arcs, y_true, y_classifier, decoded,
#                                     arc_endpoints, "student_trip_prediction.html")
# display(prediction_map)


## 8. Notes for report sections 5 and 6

Record actual choices and measured results after running on permitted data. Explain hidden states, emission probabilities, transition connectivity/distance, handling of gaps and missing candidates, and cases where decoding helped or hurt. Include the real comparison table, per-trip or aggregate metrics, candidate miss rate, and a Folium map generated by the implementation. Tune decoder settings using training and student-test data only; keep evaluator data untouched.